# Adam
Adam by hand, why the bias correction is needed, five optimizers on the students data,
and six optimizers on MNIST.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

tf.config.experimental.enable_op_determinism()   # same numbers on every run
np.set_printoptions(precision=4, suppress=True)

## Two Adam steps by hand: gradients 2 then 1, eta 0.001, beta1 0.9, beta2 0.999

In [ ]:
eta, b1, b2, eps = 0.001, 0.9, 0.999, 1e-8
m = v = 0.0
for t, g in enumerate([2.0, 1.0], start=1):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g ** 2
    m_hat, v_hat = m / (1 - b1 ** t), v / (1 - b2 ** t)
    step = eta * m_hat / (np.sqrt(v_hat) + eps)
    no_corr = eta * m / (np.sqrt(v) + eps)
    print(f"t={t}: m={m:.4f} v={v:.6f}  m_hat={m_hat:.4f} v_hat={v_hat:.4f}  step={step:.6f}"
          f"  (without correction: {no_corr:.6f})")

## Why divide by 1 - beta^t: the EWMA's weights started at 0 add up to 1 - beta^t

In [ ]:
for beta in (0.9, 0.999):
    for t in (1, 10, 100, 1000):
        weights = (1 - beta) * beta ** np.arange(t)
        print(f"beta {beta}, t {t:4d}: sum of weights {weights.sum():.4f} = 1 - beta^t = {1 - beta**t:.4f}")

A noisy gradient with mean 1 and standard deviation 0.5 (true second moment 1 + 0.25 = 1.25).

In [ ]:
g = np.random.default_rng(0).normal(1.0, 0.5, 100)
for beta, vals, true in ((0.9, g, 1.0), (0.999, g ** 2, 1.25)):
    a, raw = 0.0, []
    for x in vals:
        a = beta * a + (1 - beta) * x
        raw.append(a)
    raw = np.array(raw)
    t = np.arange(1, 101)
    print(f"beta {beta}: raw EWMA at steps 1/10/100 {raw[[0, 9, 99]].round(3)}, corrected "
          f"{(raw / (1 - beta ** t))[[0, 9, 99]].round(3)}, true value {true}")

## Five optimizers on the students data (IIT sparse, package target) from (m, b) = (-4, -4)

In [ ]:
d = pd.read_csv("data/students.csv")
X = np.c_[d.iit, np.ones(len(d))]
y = d.package.to_numpy()
best = np.linalg.lstsq(X, y, rcond=None)[0]
loss = lambda p: np.mean((y - X @ p) ** 2)
grad = lambda p: -2 * X.T @ (y - X @ p) / len(y)

def run(kind, eta, steps=300):
    p, m, v = np.array([-4.0, -4.0]), np.zeros(2), np.zeros(2)
    path = [p.copy()]
    for t in range(1, steps + 1):
        g = grad(p)
        if kind == "gd":
            p = p - eta * g
        elif kind == "momentum":
            m = 0.9 * m + eta * g
            p = p - m
        elif kind == "adagrad":
            v = v + g ** 2
            p = p - eta * g / (np.sqrt(v) + 1e-8)
        elif kind == "rmsprop":
            v = 0.9 * v + 0.1 * g ** 2
            p = p - eta * g / (np.sqrt(v) + 1e-8)
        else:
            m = 0.9 * m + 0.1 * g
            v = 0.999 * v + 0.001 * g ** 2
            p = p - eta * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)
        path.append(p.copy())
    return np.array(path)

for kind, eta in (("gd", 0.3), ("momentum", 0.1), ("adagrad", 2.0), ("rmsprop", 0.3), ("adam", 0.5),
                  ("adam", 0.1), ("adam", 0.3), ("adam", 1.0)):
    P = run(kind, eta)
    gap = np.array([loss(p) for p in P]) - loss(best)
    reached = int(np.argmax(gap < 0.01)) if (gap < 0.01).any() else None
    print(f"{kind:8s} eta {eta:<4}: within 0.01 after {reached} steps; largest distance from the best in the "
          f"last 50 steps {np.abs(P[-50:] - best).max():.3f}")

## MNIST: six optimizers
10,000 training images, the 10,000 test images for validation, batch size 64, 20 epochs, 3 seeds.
Learning rates: 0.01 for SGD, momentum and NAG (beta 0.9); 0.01 for AdaGrad; 0.001 for RMSProp and Adam
(Keras' default for both).

In [ ]:
(x, yy), (x_test, y_test) = keras.datasets.mnist.load_data()
x = x[:10000].reshape(-1, 784).astype("float32") / 255
yy = yy[:10000]
x_test = x_test.reshape(-1, 784).astype("float32") / 255
O = keras.optimizers
optimizers = {"SGD": lambda: O.SGD(0.01), "momentum": lambda: O.SGD(0.01, momentum=0.9),
              "NAG": lambda: O.SGD(0.01, momentum=0.9, nesterov=True), "AdaGrad": lambda: O.Adagrad(0.01),
              "RMSProp": lambda: O.RMSprop(0.001), "Adam": lambda: O.Adam(0.001)}

def train(make, seed, epochs=20):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([keras.Input(shape=(784,)),
                              keras.layers.Dense(128, activation="relu"),
                              keras.layers.Dense(64, activation="relu"),
                              keras.layers.Dense(10, activation="softmax")])
    model.compile(optimizer=make(), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model.fit(x, yy, epochs=epochs, batch_size=64, validation_data=(x_test, y_test), verbose=0).history

rows = []
for name, make in optimizers.items():
    for seed in range(3):
        h = train(make, seed)
        for e in range(20):
            rows.append(dict(optimizer=name, seed=seed, epoch=e + 1, loss=h["loss"][e],
                             val_accuracy=h["val_accuracy"][e]))
    print(name, "done")
runs = pd.DataFrame(rows)
runs.to_csv("data/mnist_all.csv", index=False)
mean = runs.groupby(["optimizer", "epoch"])[["loss", "val_accuracy"]].mean()
table = pd.DataFrame({name: [mean.loc[(name, 1), "loss"], mean.loc[(name, 5), "loss"], mean.loc[(name, 20), "loss"],
                             mean.loc[(name, 20), "val_accuracy"], mean.loc[name, "val_accuracy"].max()]
                      for name in optimizers},
                     index=["loss epoch 1", "loss epoch 5", "loss epoch 20", "val accuracy epoch 20", "best val accuracy"])
print(table.round(4))